# World Capitals Weather Lookup
This notebook retrieves weather data for world capitals using the OpenWeather API,  
saves the results to a CSV file, and lets the user check any capital they want.


In [10]:
import requests
import pandas as pd
import time


## Step 1 – Retrieve all capitals with their country codes
We use the REST Countries API (or a fallback CSV mirror) to build a list of capitals.


In [12]:
url = "https://restcountries.com/v3.1/all?fields=name,capital,cca2"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    countries = response.json()
    capitals = [
        (c["capital"][0], c["cca2"])
        for c in countries
        if isinstance(c, dict) and c.get("capital") and c.get("cca2")
    ]
    print(f" Retrieved {len(capitals)} capitals.")
except Exception as e:
    print("REST API failed, switching to fallback source:", e)
    csv_url = "https://raw.githubusercontent.com/dr5hn/countries-states-cities-database/master/countries.csv"
    df_c = pd.read_csv(csv_url)
    capitals = list(zip(df_c["capital"], df_c["iso2"]))
    print(f" Retrieved {len(capitals)} capitals from CSV fallback.")


 Retrieved 246 capitals.


## Step 2 – Collect weather data
We fetch current weather for each capital using the OpenWeather API.


In [18]:
api_key = "b8e520dbd7c685b995f7756cdf09d49a"
data_list = []
success_count = 0

for city, code in capitals:
    try:
        url = f"https://api.openweathermap.org/data/2.5/weather?q={city},{code}&appid={api_key}&units=metric"
        r = requests.get(url, timeout=10)
        j = r.json()
        if r.status_code == 200 and "main" in j:
            now = pd.Timestamp.now()
            data_list.append({
                "Date": now.strftime("%Y-%m-%d"),
                "Time": now.strftime("%H:%M:%S"),
                "City": city,
                "Country": code,
                "Temperature (°C)": j["main"]["temp"],
                "Humidity (%)": j["main"]["humidity"],
                "Pressure (hPa)": j["main"]["pressure"],
                "Wind (m/s)": j.get("wind", {}).get("speed"),
                "Weather": j["weather"][0]["main"]
            })
            success_count += 1
    except Exception:
        pass
    time.sleep(1.1)

print(success_count)



238


## Step 3 – Save data and let the user query one capital
We store the dataset, then prompt the user to type a capital name to display its weather.


In [19]:
df = pd.DataFrame(data_list)

if df.empty:
    print("\n No weather data was collected. Please check your API key or internet connection.")
else:
    df.to_csv("world_capitals_weather.csv", index=False)
    print("\n Weather data has been saved to 'world_capitals_weather.csv'")
    print(f" Number of capitals successfully fetched: {success_count}")

    if "City" in df.columns:
        capital_to_check = input("\nEnter the capital name to view its weather data: ").strip()
        result = df[df["City"].str.lower() == capital_to_check.lower()]
        if not result.empty:
            print(f"\n Weather data for {capital_to_check}:")
            print(result.to_string(index=False))
        else:
            print(f"\n No data found for '{capital_to_check}'. Please check the spelling and try again.")
    else:
        print("\n Column 'City' not found in the dataset. Please verify your data structure.")



 Weather data has been saved to 'world_capitals_weather.csv'
 Number of capitals successfully fetched: 238

Enter the capital name to view its weather data: Riyadh

 Weather data for Riyadh:
      Date     Time   City Country  Temperature (°C)  Humidity (%)  Pressure (hPa)  Wind (m/s) Weather
2025-10-10 06:50:56 Riyadh      SA             32.59            11            1015        1.72   Clear
